## MNIST → HLS4ML
Converts `mnist_finn_ready.onnx` to a Vivado HLS project via hls4ml.

In [1]:
import onnx
from onnx import helper, numpy_helper, TensorProto
import hls4ml
import shutil, os
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.util.to_channels_last import to_channels_last

# hls4ml bug: InferPrecisionTypes.match() calls self.inputs[0] unconditionally,
# but Constant layers (weights) always have inputs=[]. Patch it to also match
# constants so _infer_default_type() resolves their UnspecifiedPrecisionType.
import hls4ml.model.optimizer.passes.infer_precision as _ip
from hls4ml.model.types import UnspecifiedPrecisionType as _UPT

_orig_ip_match = _ip.InferPrecisionTypes.match

def _safe_ip_match(self, node):
    if not node.inputs:
        return any(isinstance(lt.precision, _UPT) for lt in node.types.values())
    return _orig_ip_match(self, node)

_ip.InferPrecisionTypes.match = _safe_ip_match
print('Imports OK')

Imports OK


### Model preprocessing helpers

In [2]:
import re

def _sanitize(name):
    """Make a name safe for C++ identifiers and hls4ml layer names.
    Replaces '.' and '/' with '_', collapses repeated underscores,
    and strips leading underscores."""
    s = name.replace('.', '_').replace('/', '_')
    s = re.sub(r'_+', '_', s)   # collapse __  ->  _
    s = s.strip('_')             # remove leading/trailing _
    return s


def _build_quant_redirect(graph):
    redirect = {}
    for node in graph.node:
        if node.op_type in ('Quant', 'IntQuant', 'BipolarQuant'):
            redirect[node.output[0]] = node.input[0]
    changed = True
    while changed:
        changed = False
        for k, v in list(redirect.items()):
            if v in redirect:
                redirect[k] = redirect[v]; changed = True
    return redirect


def _apply_redirect(graph, redirect):
    for node in graph.node:
        for i, inp in enumerate(node.input):
            if inp in redirect:
                node.input[i] = redirect[inp]
    for out in graph.output:
        if out.name in redirect:
            out.name = redirect[out.name]


def _strip_quant_nodes(graph):
    keep = [n for n in graph.node if n.op_type not in {'Quant', 'IntQuant', 'BipolarQuant'}]
    del graph.node[:]
    graph.node.extend(keep)


def _normalize_names(graph):
    """Sanitize every tensor and node name so they are valid C++ identifiers.
    hls4ml uses node names directly in typedef/variable names, so '/' and '.'
    must be removed before codegen."""
    # Collect every name that appears anywhere in the graph
    all_names = set()
    for node in graph.node:
        all_names.update(node.input)
        all_names.update(node.output)
        all_names.add(node.name)
    for inp  in graph.input:       all_names.add(inp.name)
    for out  in graph.output:      all_names.add(out.name)
    for vi   in graph.value_info:  all_names.add(vi.name)
    for init in graph.initializer: all_names.add(init.name)

    rename = {n: _sanitize(n) for n in all_names
              if n and ('.' in n or '/' in n or n != _sanitize(n))}
    if not rename:
        return

    for node in graph.node:
        node.name = rename.get(node.name, node.name)
        for i, inp in enumerate(node.input):
            if inp in rename: node.input[i] = rename[inp]
        for i, out in enumerate(node.output):
            if out in rename: node.output[i] = rename[out]
    for inp  in graph.input:       inp.name  = rename.get(inp.name,  inp.name)
    for out  in graph.output:      out.name  = rename.get(out.name,  out.name)
    for vi   in graph.value_info:  vi.name   = rename.get(vi.name,   vi.name)
    for init in graph.initializer: init.name = rename.get(init.name, init.name)


def _propagate_all_shapes(graph):
    """Manually propagate shapes -- ONNX shape_inference fails on channels-last Conv."""
    shape_map = {}; dtype_map = {}
    for info in list(graph.input) + list(graph.value_info) + list(graph.output):
        tt = info.type.tensor_type
        if tt.HasField('shape'):
            dims = [d.dim_value for d in tt.shape.dim]
            if all(d >= 0 for d in dims):
                shape_map[info.name] = dims
                dtype_map[info.name] = tt.elem_type

    def add(name, shape, et=TensorProto.FLOAT):
        if name and name not in shape_map and shape and all(d > 0 for d in shape):
            shape_map[name] = shape; dtype_map[name] = et
            ti = helper.make_tensor_type_proto(elem_type=et, shape=shape)
            graph.value_info.append(helper.make_value_info(name, ti))

    def ga(node, name, default=None):
        for a in node.attribute:
            if a.name == name:
                if a.type == 7: return list(a.ints)
                if a.type == 1: return a.f
                if a.type == 2: return a.i
        return default

    for node in graph.node:
        op  = node.op_type
        out = node.output[0] if node.output else None
        ins = [shape_map.get(i) for i in node.input]
        et  = dtype_map.get(node.input[0], TensorProto.FLOAT) if node.input else TensorProto.FLOAT

        if op in ('Relu', 'Sigmoid', 'Tanh', 'LeakyRelu', 'Elu', 'Selu', 'BatchNormalization'):
            if ins[0]: add(out, ins[0], et)

        elif op == 'Conv':  # NHWC: input [N,H,W,Cin]  weight [Cout,kH,kW,Cin]
            if ins[0] and ins[1]:
                N, H, W = ins[0][0], ins[0][1], ins[0][2]
                C_out, kH, kW = ins[1][0], ins[1][1], ins[1][2]
                pads      = ga(node, 'pads',      [0, 0, 0, 0])
                strides   = ga(node, 'strides',   [1, 1])
                dilations = ga(node, 'dilations', [1, 1])
                H_out = (H + pads[0] + pads[2] - dilations[0]*(kH-1) - 1) // strides[0] + 1
                W_out = (W + pads[1] + pads[3] - dilations[1]*(kW-1) - 1) // strides[1] + 1
                add(out, [N, H_out, W_out, C_out])

        elif op == 'MaxPool':
            if ins[0]:
                N, H, W, C = ins[0]
                kernel  = ga(node, 'kernel_shape', [1, 1])
                strides = ga(node, 'strides',      [1, 1])
                pads    = ga(node, 'pads',         [0, 0, 0, 0])
                H_out = (H + pads[0] + pads[2] - kernel[0]) // strides[0] + 1
                W_out = (W + pads[1] + pads[3] - kernel[1]) // strides[1] + 1
                add(out, [N, H_out, W_out, C])

        elif op == 'Flatten':
            if ins[0]:
                axis = ga(node, 'axis', 1)
                N = ins[0][0]; flat = 1
                for d in ins[0][axis:]: flat *= d
                add(out, [N, flat])

        elif op == 'MatMul':
            if ins[0] and ins[1]:
                add(out, ins[0][:-1] + [ins[1][-1]])

        elif op == 'Add':
            s0 = ins[0]; s1 = ins[1] if len(ins) > 1 else None
            if s0 and s1: add(out, s0 if len(s0) >= len(s1) else s1)
            elif s0:      add(out, s0)

print('Helpers defined')

Helpers defined


### Prepare model

In [ ]:
import tempfile, os

def prepare_model_for_hls4ml(model_path):
    tmp = tempfile.gettempdir()
    shaped_path = os.path.join(tmp, '_hls4ml_shaped.onnx')
    cl_path     = os.path.join(tmp, '_hls4ml_cl.onnx')

    # Step 0: infer shapes, convert to channels-last (handles both NCHW and NHWC)
    mw = ModelWrapper(model_path)
    mw = mw.transform(InferShapes())
    mw.save(shaped_path)
    to_channels_last(shaped_path, make_input_channels_last=True,
                     out_file=cl_path)
    model = onnx.load(cl_path)
    graph = model.graph

    # Fix 1: give unnamed nodes a name
    for i, node in enumerate(graph.node):
        if not node.name:
            node.name = f'{node.op_type}_{i}'

    # Fix 2: strip Quant nodes
    redirect = _build_quant_redirect(graph)
    _apply_redirect(graph, redirect)
    _strip_quant_nodes(graph)

    # Fix 3: remove orphan initializers (Quant scale/zp params, now unreferenced)
    referenced = {inp for node in graph.node for inp in node.input}
    keep = [init for init in graph.initializer if init.name in referenced]
    del graph.initializer[:]
    graph.initializer.extend(keep)

    # Fix 4: sanitize ALL names -- replace '.' and '/' with '_' so names are
    # valid C++ identifiers (hls4ml writes node names directly into typedefs)
    _normalize_names(graph)

    # Fix 5: Gemm -> MatMul+Add (hls4ml doesn't support Gemm)
    init_map = {init.name: init for init in graph.initializer}
    for node in graph.node:
        if node.op_type != 'Gemm': continue
        attrs = {a.name: a for a in node.attribute}
        if not (attrs.get('transB') and attrs['transB'].i): continue
        src = init_map.get(node.input[1])
        if src is not None:
            arr = numpy_helper.to_array(src).T
            new_init = numpy_helper.from_array(arr, name=src.name)
            graph.initializer.remove(src)
            graph.initializer.append(new_init)
            init_map[src.name] = new_init
    new_nodes = []
    for node in graph.node:
        if node.op_type != 'Gemm':
            new_nodes.append(node); continue
        A, B, C = node.input[0], node.input[1], node.input[2]
        Y = node.output[0]; p = node.name or Y; mm = p + '_mm'
        new_nodes.append(helper.make_node('MatMul', [A, B], [mm], name=p + '_matmul'))
        new_nodes.append(helper.make_node('Add',    [mm, C], [Y], name=p + '_add'))
    del graph.node[:]
    graph.node.extend(new_nodes)

    # Fix 6: rebuild graph.input with initializer shapes so hls4ml can find weights
    del graph.value_info[:]
    existing = {inp.name for inp in graph.input}
    for init in graph.initializer:
        if init.name not in existing:
            ti = helper.make_tensor_type_proto(elem_type=init.data_type, shape=list(init.dims))
            graph.input.append(helper.make_value_info(init.name, ti))

    # Fix 7: manually propagate shapes
    _propagate_all_shapes(graph)

    onnx.checker.check_model(model)
    return model


model = prepare_model_for_hls4ml('mnist_finn_ready.onnx')
print('Model ready. Ops:', [n.op_type for n in model.graph.node])
print('Node names:', [n.name for n in model.graph.node])

### Generate hls4ml config

In [ ]:
config = hls4ml.utils.config_from_onnx_model(model, granularity='name', backend='Vivado')
config['Model']['Precision']   = 'ap_fixed<8,4>'
config['Model']['ReuseFactor'] = 32
config['Model']['Strategy']    = 'Resource'

# Propagate model-level ReuseFactor to all layers so per-layer defaults don't override it
for layer in config.get('LayerName', {}).values():
    layer['ReuseFactor'] = 32

print('Config ready. Layers:')
for name in config.get('LayerName', {}):
    print(' ', name)

### Convert and write HLS project

In [5]:
if os.path.exists('my_hls_project'):
    shutil.rmtree('my_hls_project')

hls_model = hls4ml.converters.convert_from_onnx_model(
    model,
    hls_config=config,
    output_dir='my_hls_project',
    backend='Vivado',
    part='xc7z020clg484-1',
    clock_period=10,
    io_type='io_stream',
)
hls_model.write()

print('HLS project written to: my_hls_project/')
print('Contents:', sorted(os.listdir('my_hls_project')))

Interpreting Model ...
Output layers:  ['fc2_Gemm_add']
Input shape: [28, 28, 1]
Topology:
Layer name: conv1_Conv, layer type: Conv, current shape: [[1, 28, 28, 1], [32, 3, 3, 1], [32]]
Layer name: relu1_act_quant_activation_impl_Relu, layer type: Activation, current shape: [[1, 28, 28, 32]]
Layer name: pool_MaxPool, layer type: MaxPooling2D, current shape: [[1, 28, 28, 32]]
Layer name: conv2_Conv, layer type: Conv, current shape: [[1, 14, 14, 32], [64, 3, 3, 32], [64]]
Layer name: relu2_act_quant_activation_impl_Relu, layer type: Activation, current shape: [[1, 14, 14, 64]]
Layer name: pool2_MaxPool, layer type: MaxPooling2D, current shape: [[1, 14, 14, 64]]
Layer name: Flatten_17, layer type: Reshape, current shape: [[1, 3, 3, 64]]
Layer name: fc1_Gemm_matmul, layer type: MatMul, current shape: [[1, 576], [576, 128]]
Layer name: fc1_Gemm_add, layer type: Merge, current shape: [[1, 128], [128]]
Layer name: relu3_act_quant_activation_impl_Relu, layer type: Activation, current shape: [[

### Synthesize (requires Vivado HLS on PATH)

In [6]:
# C-simulation only (fast check, no FPGA needed):
# hls_model.build(csim=True, synth=False)

# Full synthesis:
# hls_model.build(csim=False, synth=True, export=True)